# R.A.G Pipeline

In [1]:
from chromadb.config import Settings
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any, Tuple
import chromadb
import numpy as np
import os
import uuid

/var/folders/z6/khrbpfx14l34x1vkz0ngd7nm0000gn/T/ipykernel_5953/1381683733.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/Users/joseph/Desktop/Projects/learning/langchain_and_RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Ingestion

In [2]:
loader = PyMuPDFLoader(file_path="https://josephjohn.me/Joseph%20John%20CV.pdf")
resume = loader.load()
resume

[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'https://josephjohn.me/Joseph%20John%20CV.pdf', 'file_path': 'https://josephjohn.me/Joseph%20John%20CV.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'Joseph John CV', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Joseph John \nGalway, Ireland \nEmail: josephjohn2k00@gmail.com | Mobile: +353 8945 08344 | LinkedIn | GitHub \nPERSONAL PROFILE \nSoftware Engineer and AWS Certified Cloud Practitioner with hands-on experience building scalable, cloud-native \nbackend services using Python and AWS serverless technologies (Lambda, API Gateway, DynamoDB). Strong \nfoundation in RESTful API design, microservices architecture, and object-oriented development. Recently completed \na 1st Class Honours MSc in Information Systems Management with a focus on Cloud Computing, eager to grow as a

## Chunking

In [3]:
def split_doc(documents, chunk_size=1000, chunk_overlap=200):
    txt_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    return txt_splitter.split_documents(documents)

In [4]:
chunks = split_doc(resume)
chunks

[Document(metadata={'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'https://josephjohn.me/Joseph%20John%20CV.pdf', 'file_path': 'https://josephjohn.me/Joseph%20John%20CV.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'Joseph John CV', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Joseph John \nGalway, Ireland \nEmail: josephjohn2k00@gmail.com | Mobile: +353 8945 08344 | LinkedIn | GitHub \nPERSONAL PROFILE \nSoftware Engineer and AWS Certified Cloud Practitioner with hands-on experience building scalable, cloud-native \nbackend services using Python and AWS serverless technologies (Lambda, API Gateway, DynamoDB). Strong \nfoundation in RESTful API design, microservices architecture, and object-oriented development. Recently completed \na 1st Class Honours MSc in Information Systems Management with a focus on Cloud Computing, eager to grow as a

## Embedding

#### Create embedding manager

In [5]:
class EmbeddingManager:
    """Handles document embeddings using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialise the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            self.model = SentenceTransformer(self.model_name)
        except Exception as e:
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate Embeddings for the list of texts

        Args:
            texts: List of text strings to embed
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dimension)
        """
        if not self.model:
            raise ValueError("Model Not Loaded")

        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

## Initialize embedding manager
embedding_manager = EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9762.35it/s]


#### Create embeddings

In [6]:
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts) 

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.03it/s]


## Vector Store

#### Initialise ChromaDB Vector Store

In [7]:
class VectorStore:
    """Manages embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "resume_data", persistent_directory: str = "../data/vector_store"):
        """
        Initializes vector store

        Args:
            collection_name: Name of ChromaDB collection
            persistent_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            # Create client
            os.makedirs(self.persistent_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistent_directory)

            # Get/Create a collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Resume embeddings for RAG"}
            )
        except Exception as e:
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain Documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embeddings")


        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        document_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate uuid
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            document_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_text
            )
        except Exception as e:
            print(f"Error adding vectors to store: {e}")
            raise

# Initialise Vector Store
vs=VectorStore()

#### Add embeddings to Vector Store

In [8]:
vs.add_documents(chunks, embeddings)